In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install pyedflib

In [ ]:
# =============================================================================
#  APNEA DATASET BUILDER — SN1 to SN100
#  - float16 storage  → ~9 GB total  (fits Kaggle 19.5 GB disk)
#  - One patient in RAM at a time
#  - Auto-resume after crash
#  - No X_all.npy (never creates 18 GB file)
#  - PyTorch Dataset + DataLoaders ready for P100 GPU
# =============================================================================
#this is main dataset creater
# ──────────────────────────────────────────────────────────────────────────────
# 0.  DOWNLOAD
# ──────────────────────────────────────────────────────────────────────────────
import kagglehub
path = kagglehub.dataset_download("nishantomhare/nimhanspatient-dataset")
print("Dataset path:", path)

# ──────────────────────────────────────────────────────────────────────────────
# 1.  IMPORTS
# ──────────────────────────────────────────────────────────────────────────────
import os, gc, math, warnings
import numpy as np
import pandas as pd
import pyedflib
import torch
from torch.utils.data import Dataset, DataLoader
from scipy.signal import resample
from collections import Counter

warnings.filterwarnings("ignore")

# ──────────────────────────────────────────────────────────────────────────────
# 2.  CONFIGURATION
# ──────────────────────────────────────────────────────────────────────────────
EDF_DIR    = path
XLSX_DIR   = path
OUTPUT_DIR = "/kaggle/working/dataset_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET_FS          = 200
WINDOW_SEC         = 30
SAMPLES_PER_WINDOW = WINDOW_SEC * TARGET_FS        # 6000
SHEET_NAME         = "Flow Events"
SKIP_ROWS          = 5

FIXED_CHANNEL_KEYS = ["flow","chest","abd","spo2","pleth","ecg","pulse","flow2"]
NUM_CHANNELS       = len(FIXED_CHANNEL_KEYS)       # 8

CHANNEL_MAP = {
    "flow"  : ["Pressure Flow", "Airflow"],
    "chest" : ["Thorax", "Chest", "RIP Thorax"],
    "abd"   : ["RIP Abdomen", "Abdomen"],
    "spo2"  : ["SpO2", "SPO2", "SaO2"],
    "pleth" : ["Pleth", "Plethysmography"],
    "ecg"   : ["ECG 2", "ECG", "EKG"],
    "pulse" : ["Pulse", "Pulse Rate"],
    "flow2"   : ["Flow Th"],
}

APNEA_KEYWORDS = ["apnea", "hypopnea","RERA"]

edf_files   = sorted([f for f in os.listdir(EDF_DIR) if f.lower().endswith(".edf")])
PATIENT_IDS = [os.path.splitext(f)[0] for f in edf_files]

print(f"\nFound {len(PATIENT_IDS)} EDF files")
print(f"Patients : {PATIENT_IDS}")

# ──────────────────────────────────────────────────────────────────────────────
# 3.  HELPERS
# ──────────────────────────────────────────────────────────────────────────────
def time_to_sec(t) -> float:
    return t.hour * 3600 + t.minute * 60 + t.second

def sec_to_hms(total_sec: float) -> str:
    total_sec = int(total_sec) % 86400
    h, r = divmod(total_sec, 3600)
    m, s = divmod(r, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

def next_aligned_second(start_sec: float, window: int) -> float:
    return math.ceil(start_sec / window) * window

def already_processed(pid: str) -> bool:
    return all(os.path.exists(os.path.join(OUTPUT_DIR, f"{p}_{pid}.{e}"))
               for p, e in [("X","npy"),("y","npy"),("meta","csv")])

def disk_free_gb() -> float:
    st = os.statvfs(OUTPUT_DIR)
    return st.f_bavail * st.f_frsize / 1e9

# ──────────────────────────────────────────────────────────────────────────────
# 4.  CONVERT EXISTING float32 FILES TO float16  (run once if needed)
#     Frees ~9 GB if you already have float32 files from a previous run
# ──────────────────────────────────────────────────────────────────────────────
def convert_existing_to_float16():
    """
    Convert any existing float32 X_SN*.npy files to float16 in-place.
    Deletes float32 original immediately after saving float16 copy.
    Safe to re-run — skips files already in float16.
    """
    x_files = sorted([f for f in os.listdir(OUTPUT_DIR)
                       if f.startswith("X_SN") and f.endswith(".npy")])
    if not x_files:
        return

    print("\n" + "=" * 55)
    print("  Checking existing X files for float32 → float16 conversion")
    print("=" * 55)

    for fname in x_files:
        fpath = os.path.join(OUTPUT_DIR, fname)
        X     = np.load(fpath, mmap_mode="r")

        if X.dtype == np.float16:
            print(f"  ✅ {fname} already float16 — skipped")
            del X
            continue

        print(f"  🔄 {fname}  {X.dtype} → float16 ...", end=" ")
        tmp = fpath + ".tmp"
        np.save(tmp, X.astype(np.float16))
        del X
        gc.collect()
        os.remove(fpath)
        os.rename(tmp, fpath)
        size_mb = os.path.getsize(fpath) / 1e6
        print(f"done  ({size_mb:.0f} MB)  disk free: {disk_free_gb():.1f} GB")

convert_existing_to_float16()

# ──────────────────────────────────────────────────────────────────────────────
# 5.  PER-PATIENT PROCESSING
# ──────────────────────────────────────────────────────────────────────────────
def process_and_save_patient(pid: str):
    """
    Build windows for one patient, save float16 X + int8 y + meta CSV.
    Returns (n_windows, n_apnea, n_normal) or None.
    """
    edf_path  = os.path.join(EDF_DIR,  f"{pid}.edf")
    xlsx_path = os.path.join(XLSX_DIR, f"{pid}.xlsx")

    if not os.path.exists(edf_path):
        print(f"  ⚠️  [{pid}] EDF not found  → skipped")
        return None
    if not os.path.exists(xlsx_path):
        print(f"  ⚠️  [{pid}] XLSX not found → skipped")
        return None

    # ── check disk space before starting ─────────────────────────
    if disk_free_gb() < 0.5:
        print(f"  ❌ [{pid}] Less than 0.5 GB free — stopping to avoid crash")
        return None

    # ── load EDF ─────────────────────────────────────────────────
    try:
        f             = pyedflib.EdfReader(edf_path)
        ch_labels     = f.getSignalLabels()
        ch_lower      = {c.lower(): c for c in ch_labels}
        edf_start     = pd.to_datetime(f.getStartdatetime()).time()
        edf_start_sec = time_to_sec(edf_start)

        raw_data = {}
        for key, candidates in CHANNEL_MAP.items():
            for name in candidates:
                match = name if name in ch_labels else ch_lower.get(name.lower())
                if match:
                    idx = ch_labels.index(match)
                    raw_data[key] = (f.readSignal(idx),
                                     int(f.getSampleFrequency(idx)))
                    break
        f.close()
    except Exception as e:
        print(f"  ❌ [{pid}] EDF error: {e} → skipped")
        return None

    if not raw_data:
        print(f"  ⚠️  [{pid}] No channels matched → skipped")
        return None

    # ── resample to 200 Hz ───────────────────────────────────────
    max_time      = max(len(s) / fs for s, fs in raw_data.values())
    target_length = int(max_time * TARGET_FS)
    resampled     = {}

    for key, (sig, fs) in raw_data.items():
        if fs == TARGET_FS:
            res = sig.copy()
        else:
            res = resample(sig, int(len(sig) * TARGET_FS / fs))
        if len(res) < target_length:
            res = np.pad(res, (0, target_length - len(res)), mode="edge")
        else:
            res = res[:target_length]
        resampled[key] = res.astype(np.float32)

    del raw_data
    gc.collect()

    # ── fixed 8 channels, zeros for missing ──────────────────────
    fixed       = {}
    missing_chs = []
    for key in FIXED_CHANNEL_KEYS:
        if key in resampled:
            fixed[key] = resampled[key]
        else:
            fixed[key] = np.zeros(target_length, dtype=np.float32)
            missing_chs.append(key)

    del resampled
    gc.collect()

    if missing_chs:
        print(f"  ⚠️  [{pid}] Missing → zero-padded: {missing_chs}")

    # ── load XLSX events ─────────────────────────────────────────
    try:
        df = pd.read_excel(xlsx_path, sheet_name=SHEET_NAME, skiprows=SKIP_ROWS)
        df.columns = ["start","end","duration","event","stage"] + list(df.columns[5:])
        df = df[df["event"].notna() & (df["event"] != "Event name")].reset_index(drop=True)
        df["start"] = pd.to_datetime(df["start"], errors="coerce").dt.time
        df["end"]   = pd.to_datetime(df["end"],   errors="coerce").dt.time
        df = df.dropna(subset=["start","end"])

        df["start_sec"] = df["start"].apply(time_to_sec)
        df["end_sec"]   = df["end"].apply(time_to_sec)
        df.loc[df["start_sec"] < edf_start_sec, "start_sec"] += 86400
        df.loc[df["end_sec"]   < edf_start_sec, "end_sec"]   += 86400
        df["start_rel"] = df["start_sec"] - edf_start_sec
        df["end_rel"]   = df["end_sec"]   - edf_start_sec

        mask   = df["event"].str.contains("|".join(APNEA_KEYWORDS), case=False, na=False)
        events = df[mask].copy().reset_index(drop=True)
        del df
    except Exception as e:
        print(f"  ❌ [{pid}] XLSX error: {e} → skipped")
        return None

    ev_start = events["start_rel"].to_numpy()
    ev_end   = events["end_rel"].to_numpy()

    def label_window(ws, we):
        if len(ev_start) == 0:
            return 0
        return int(np.any(~((ev_end <= ws) | (ev_start >= we))))

    # ── build windows ────────────────────────────────────────────
    total_dur = target_length / TARGET_FS
    offset    = next_aligned_second(edf_start_sec, WINDOW_SEC) - edf_start_sec
    n_est     = max(0, int((total_dur - offset) // WINDOW_SEC))

    if n_est == 0:
        print(f"  ⚠️  [{pid}] No windows possible → skipped")
        return None

    # float16 pre-allocation — half the RAM of float32
    X_pat     = np.zeros((n_est, NUM_CHANNELS, SAMPLES_PER_WINDOW), dtype=np.float16)
    y_pat     = np.zeros(n_est, dtype=np.int8)
    meta_rows = []
    actual    = 0
    current   = offset

    while current + WINDOW_SEC <= total_dur:
        s    = int(current * TARGET_FS)
        e    = s + SAMPLES_PER_WINDOW
        segs = [fixed[k][s:e] for k in FIXED_CHANNEL_KEYS]

        if all(len(sg) == SAMPLES_PER_WINDOW for sg in segs):
            lbl           = label_window(current, current + WINDOW_SEC)
            # convert to float16 at assignment — no extra copy
            X_pat[actual] = np.stack(segs, axis=0).astype(np.float16)
            y_pat[actual] = lbl
            meta_rows.append({
                "patient"     : pid,
                "window_idx"  : actual,
                "start_rel_s" : current,
                "end_rel_s"   : current + WINDOW_SEC,
                "start_clock" : sec_to_hms(edf_start_sec + current),
                "end_clock"   : sec_to_hms(edf_start_sec + current + WINDOW_SEC),
                "label"       : lbl,
            })
            actual += 1

        current += WINDOW_SEC

    X_pat = X_pat[:actual]
    y_pat = y_pat[:actual]

    if actual == 0:
        print(f"  ⚠️  [{pid}] No valid windows → skipped")
        del X_pat, y_pat, fixed, events
        gc.collect()
        return None

    # ── save to disk immediately then free ───────────────────────
    np.save(os.path.join(OUTPUT_DIR, f"X_{pid}.npy"), X_pat)   # float16
    np.save(os.path.join(OUTPUT_DIR, f"y_{pid}.npy"), y_pat)   # int8
    pd.DataFrame(meta_rows).to_csv(
        os.path.join(OUTPUT_DIR, f"meta_{pid}.csv"), index=False)

    dist  = Counter(y_pat.tolist())
    total = actual
    size_mb = os.path.getsize(os.path.join(OUTPUT_DIR, f"X_{pid}.npy")) / 1e6
    print(f"  ✅ [{pid}]  "
          f"dur={max_time/60:.0f}min  events={len(events):3d}  "
          f"windows={total:4d}  "
          f"normal={dist.get(0,0):4d}({100*dist.get(0,0)/total:.0f}%)  "
          f"apnea={dist.get(1,0):4d}({100*dist.get(1,0)/total:.0f}%)  "
          f"size={size_mb:.0f}MB  disk_free={disk_free_gb():.1f}GB"
          + (f"  zeros={missing_chs}" if missing_chs else ""))

    del X_pat, y_pat, fixed, events
    gc.collect()

    return total, dist.get(1,0), dist.get(0,0)

# ──────────────────────────────────────────────────────────────────────────────
# 6.  PHASE 1 — PROCESS ALL PATIENTS
# ──────────────────────────────────────────────────────────────────────────────
ok_pids = []
skipped = []
summary = []

print("\n" + "=" * 65)
print(f"  PHASE 1 — {len(PATIENT_IDS)} patients  (resume-safe, float16)")
print(f"  Disk free: {disk_free_gb():.1f} GB")
print("=" * 65)

for pid in PATIENT_IDS:
    if already_processed(pid):
        y_tmp = np.load(os.path.join(OUTPUT_DIR, f"y_{pid}.npy"), mmap_mode="r")
        d, n  = Counter(y_tmp.tolist()), len(y_tmp)
        print(f"  ⏭️  [{pid}] already saved — {n} windows  "
              f"normal={d.get(0,0)}  apnea={d.get(1,0)}")
        ok_pids.append(pid)
        summary.append({"patient":pid,"windows":n,
                         "apnea":d.get(1,0),"normal":d.get(0,0)})
        del y_tmp; gc.collect()
        continue

    result = process_and_save_patient(pid)
    if result is None:
        skipped.append(pid)
    else:
        n_win, n_apnea, n_normal = result
        ok_pids.append(pid)
        summary.append({"patient":pid,"windows":n_win,
                         "apnea":n_apnea,"normal":n_normal})

print(f"\n  Phase 1 complete — OK: {len(ok_pids)}  |  Skipped: {len(skipped)}")
if skipped:
    print(f"  Skipped: {skipped}")

# ──────────────────────────────────────────────────────────────────────────────
# 7.  PHASE 2 — LIGHTWEIGHT INDEX  (no X_all.npy)
# ──────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("  PHASE 2 — Building index_all.csv  (no 18 GB X_all.npy)")
print("=" * 65)

metas = []
for pid in ok_pids:
    p = os.path.join(OUTPUT_DIR, f"meta_{pid}.csv")
    if os.path.exists(p):
        metas.append(pd.read_csv(p))

index_df = pd.concat(metas, ignore_index=True)
index_df.to_csv(os.path.join(OUTPUT_DIR, "index_all.csv"), index=False)
pd.DataFrame(summary).to_csv(
    os.path.join(OUTPUT_DIR, "patient_summary.csv"), index=False)

del metas; gc.collect()

total_windows = sum(s["windows"] for s in summary)
total_apnea   = sum(s["apnea"]   for s in summary)
total_normal  = sum(s["normal"]  for s in summary)
total         = total_windows

print(f"  index_all.csv : {len(index_df):,} rows")
print(f"  Normal  (0)   : {total_normal:,}  ({100*total_normal/total:.1f}%)")
print(f"  Apnea   (1)   : {total_apnea:,}  ({100*total_apnea/total:.1f}%)")
print(f"  Disk free     : {disk_free_gb():.1f} GB")

# ──────────────────────────────────────────────────────────────────────────────
# 8.  PYTORCH DATASET
# ──────────────────────────────────────────────────────────────────────────────
class ApneaDataset(Dataset):
    """
    Reads float16 windows on demand via mmap, returns float32 tensors.
    RAM = one batch only (~6 MB for batch_size=64).
    """

    def __init__(self, output_dir, patient_ids):
        self.output_dir = output_dir
        self._cache     = {}
        self.index      = []

        for pid in patient_ids:
            yp = os.path.join(output_dir, f"y_{pid}.npy")
            xp = os.path.join(output_dir, f"X_{pid}.npy")
            if not os.path.exists(yp) or not os.path.exists(xp):
                continue
            n = len(np.load(yp, mmap_mode="r"))
            self.index.extend([(pid, i) for i in range(n)])

    def _arrays(self, pid):
        if pid not in self._cache:
            self._cache[pid] = (
                np.load(os.path.join(self.output_dir, f"X_{pid}.npy"),
                        mmap_mode="r"),   # float16 on disk
                np.load(os.path.join(self.output_dir, f"y_{pid}.npy"),
                        mmap_mode="r"),   # int8 on disk
            )
        return self._cache[pid]

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        pid, local_idx = self.index[idx]
        X_arr, y_arr   = self._arrays(pid)
        # float16 → float32 happens here (one window at a time, ~96 KB)
        x = torch.tensor(X_arr[local_idx].astype(np.float32), dtype=torch.float32)
        y = torch.tensor(int(y_arr[local_idx]), dtype=torch.long)
        return x, y

# ──────────────────────────────────────────────────────────────────────────────
# 9.  TRAIN / VAL / TEST SPLIT  (by patient — no data leakage)
# ──────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("  PHASE 3 — Train / Val / Test split")
print("=" * 65)

VAL_FRAC  = 0.15
TEST_FRAC = 0.15
SEED      = 42
BATCH     = 64
WORKERS   = 2

rng      = np.random.default_rng(SEED)
pids_arr = np.array(sorted(ok_pids))
rng.shuffle(pids_arr)

n         = len(pids_arr)
n_test    = max(1, int(n * TEST_FRAC))
n_val     = max(1, int(n * VAL_FRAC))
test_pids  = list(pids_arr[:n_test])
val_pids   = list(pids_arr[n_test:n_test+n_val])
train_pids = list(pids_arr[n_test+n_val:])

print(f"  Train : {len(train_pids)} patients → {train_pids}")
print(f"  Val   : {len(val_pids)} patients  → {val_pids}")
print(f"  Test  : {len(test_pids)} patients → {test_pids}")

train_ds = ApneaDataset(OUTPUT_DIR, train_pids)
val_ds   = ApneaDataset(OUTPUT_DIR, val_pids)
test_ds  = ApneaDataset(OUTPUT_DIR, test_pids)

print(f"\n  Train windows : {len(train_ds):,}")
print(f"  Val   windows : {len(val_ds):,}")
print(f"  Test  windows : {len(test_ds):,}")

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          num_workers=WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,
                          num_workers=WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False,
                          num_workers=WORKERS, pin_memory=True)

# ── sanity check ─────────────────────────────────────────────
print("\n  Sanity check — one batch from train_loader...")
X_b, y_b = next(iter(train_loader))
print(f"  X : {X_b.shape}  dtype={X_b.dtype}")
print(f"  y : {y_b.shape}  dtype={y_b.dtype}  unique={y_b.unique().tolist()}")

# ──────────────────────────────────────────────────────────────────────────────
# 10.  FINAL SUMMARY
# ──────────────────────────────────────────────────────────────────────────────
print(f"""
  ╔══════════════════════════════════════════════════════════╗
  ║              DATASET BUILD COMPLETE                      ║
  ╠══════════════════════════════════════════════════════════╣
  ║  Total windows   : {total_windows:<37,}║
  ║  Normal  (0)     : {total_normal:<6,}  ({100*total_normal/total:.1f}%)                  ║
  ║  Apnea   (1)     : {total_apnea:<6,}  ({100*total_apnea/total:.1f}%)                  ║
  ║  Patients OK     : {len(ok_pids):<37}║
  ║  Patients skip   : {len(skipped):<37}║
  ╠══════════════════════════════════════════════════════════╣
  ║  Train windows   : {len(train_ds):<37,}║
  ║  Val   windows   : {len(val_ds):<37,}║
  ║  Test  windows   : {len(test_ds):<37,}║
  ╠══════════════════════════════════════════════════════════╣
  ║  Storage         : float16  (~9 GB vs 18 GB float32)    ║
  ║  RAM per batch   : ~6 MB   (not 18 GB!)                 ║
  ║  No X_all.npy    : 18 GB file never created             ║
  ║  Disk free now   : {f"{disk_free_gb():.1f} GB":<37}║
  ╠══════════════════════════════════════════════════════════╣
  ║  Training:                                               ║
  ║    device = torch.device("cuda")                        ║
  ║    for X, y in train_loader:                            ║
  ║        X = X.to(device)  # (64, 8, 6000) float32       ║
  ║        y = y.to(device)  # (64,)         int64         ║
  ╚══════════════════════════════════════════════════════════╝
""")

In [ ]:
import shutil

shutil.make_archive('/kaggle/working/dataset_output', 'zip', '/kaggle/working/dataset_output')

In [ ]:
from IPython.display import FileLink
FileLink('/kaggle/working/dataset_output.zip')

In [ ]:
# =============================================================================
#  DATASET VERIFICATION — Compare saved .npy files against original EDF+XLSX
#  Checks: signal values, channel order, window timestamps, labels, shape
# =============================================================================

# ──────────────────────────────────────────────────────────────────────────────
# 0.  DOWNLOAD BOTH DATASETS
# ──────────────────────────────────────────────────────────────────────────────
import kagglehub

# original EDF + XLSX files
src_path = kagglehub.dataset_download("nishantomhare/nimhanspatient-dataset")
print("Original EDF/XLSX path :", src_path)

# saved npy dataset
npy_path = kagglehub.dataset_download("nishantomhare/detection-of-disordered-breathing-dataset2")
print("Saved NPY dataset path :", npy_path)

# ──────────────────────────────────────────────────────────────────────────────
# 1.  IMPORTS
# ──────────────────────────────────────────────────────────────────────────────
import os, gc, math, warnings
import numpy as np
import pandas as pd
import pyedflib
from scipy.signal import resample
from collections import Counter

warnings.filterwarnings("ignore")

# ──────────────────────────────────────────────────────────────────────────────
# 2.  CONFIGURATION  (must match what was used when building the dataset)
# ──────────────────────────────────────────────────────────────────────────────
TARGET_FS          = 200
WINDOW_SEC         = 30
SAMPLES_PER_WINDOW = WINDOW_SEC * TARGET_FS        # 6000
SHEET_NAME         = "Flow Events"
SKIP_ROWS          = 5

FIXED_CHANNEL_KEYS = ["flow","chest","abd","spo2","pleth","ecg","pulse","flow2"]
NUM_CHANNELS       = len(FIXED_CHANNEL_KEYS)

CHANNEL_MAP = {
    "flow"  : ["Pressure Flow", "Airflow"],
    "chest" : ["Thorax", "Chest", "RIP Thorax"],
    "abd"   : ["RIP Abdomen", "Abdomen"],
    "spo2"  : ["SpO2", "SPO2", "SaO2"],
    "pleth" : ["Pleth", "Plethysmography"],
    "ecg"   : ["ECG 2", "ECG", "EKG"],
    "pulse" : ["Pulse", "Pulse Rate"],
    "flow2"   : ["Flow Th"],
}

APNEA_KEYWORDS = ["apnea", "hypopnea","RERA"]

# ──────────────────────────────────────────────────────────────────────────────
# 3.  HELPERS
# ──────────────────────────────────────────────────────────────────────────────
def time_to_sec(t) -> float:
    return t.hour * 3600 + t.minute * 60 + t.second

def sec_to_hms(total_sec: float) -> str:
    total_sec = int(total_sec) % 86400
    h, r = divmod(total_sec, 3600)
    m, s = divmod(r, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

def next_aligned_second(start_sec: float, window: int) -> float:
    return math.ceil(start_sec / window) * window

def find_npy_file(pid, prefix, search_path):
    """Find X_{pid}.npy or y_{pid}.npy anywhere under search_path."""
    for root, dirs, files in os.walk(search_path):
        for f in files:
            if f == f"{prefix}_{pid}.npy":
                return os.path.join(root, f)
    return None

def find_meta_file(pid, search_path):
    for root, dirs, files in os.walk(search_path):
        for f in files:
            if f == f"meta_{pid}.csv":
                return os.path.join(root, f)
    return None

# ──────────────────────────────────────────────────────────────────────────────
# 4.  DISCOVER PATIENTS
# ──────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("  DISCOVERING FILES")
print("=" * 65)

# patients in original source
edf_files   = sorted([f for f in os.listdir(src_path) if f.lower().endswith(".edf")])
src_pids    = [os.path.splitext(f)[0] for f in edf_files]

# patients in saved npy dataset
npy_files   = []
for root, dirs, files in os.walk(npy_path):
    for f in files:
        if f.startswith("X_SN") and f.endswith(".npy"):
            npy_files.append(f)
npy_pids = sorted([f.replace("X_","").replace(".npy","") for f in npy_files])

print(f"  Original EDF patients : {len(src_pids)}  → {src_pids[:5]}...{src_pids[-5:]}")
print(f"  Saved NPY patients    : {len(npy_pids)}  → {npy_pids[:5]}...{npy_pids[-5:]}")

in_both   = sorted(set(src_pids) & set(npy_pids))
only_src  = sorted(set(src_pids) - set(npy_pids))
only_npy  = sorted(set(npy_pids) - set(src_pids))

print(f"\n  In both               : {len(in_both)}")
print(f"  Only in EDF (missing npy) : {only_src}")
print(f"  Only in NPY (no EDF)      : {only_npy}")

# ──────────────────────────────────────────────────────────────────────────────
# 5.  PER-PATIENT VERIFICATION FUNCTION
# ──────────────────────────────────────────────────────────────────────────────
def verify_patient(pid):
    """
    Recompute windows from scratch for this patient and compare with saved npy.
    Returns dict with pass/fail for each check.
    """
    result = {
        "pid"              : pid,
        "shape_ok"         : False,
        "n_windows_ok"     : False,
        "labels_ok"        : False,
        "signal_corr_ok"   : False,
        "timestamps_ok"    : False,
        "label_match_pct"  : 0.0,
        "signal_corr_mean" : 0.0,
        "n_saved"          : 0,
        "n_recomputed"     : 0,
        "n_label_match"    : 0,
        "error"            : "",
    }

    edf_path  = os.path.join(src_path, f"{pid}.edf")
    xlsx_path = os.path.join(src_path, f"{pid}.xlsx")
    X_path    = find_npy_file(pid, "X", npy_path)
    y_path    = find_npy_file(pid, "y", npy_path)
    meta_path = find_meta_file(pid, npy_path)

    # ── check files exist ────────────────────────────────────────
    for p, name in [(edf_path,"EDF"),(xlsx_path,"XLSX"),(X_path,"X.npy"),(y_path,"y.npy")]:
        if not p or not os.path.exists(p):
            result["error"] = f"{name} not found"
            return result

    # ── load saved npy ───────────────────────────────────────────
    X_saved = np.load(X_path, mmap_mode="r").astype(np.float32)
    y_saved = np.load(y_path, mmap_mode="r")

    result["n_saved"] = len(y_saved)

    # ── CHECK 1: shape ───────────────────────────────────────────
    expected_shape_sfx = (NUM_CHANNELS, SAMPLES_PER_WINDOW)
    if X_saved.ndim == 3 and X_saved.shape[1:] == expected_shape_sfx:
        result["shape_ok"] = True
    else:
        result["error"] = f"Shape mismatch: got {X_saved.shape}, expected (N,{NUM_CHANNELS},{SAMPLES_PER_WINDOW})"
        return result

    # ── recompute from EDF ───────────────────────────────────────
    try:
        f             = pyedflib.EdfReader(edf_path)
        ch_labels     = f.getSignalLabels()
        ch_lower      = {c.lower(): c for c in ch_labels}
        edf_start     = pd.to_datetime(f.getStartdatetime()).time()
        edf_start_sec = time_to_sec(edf_start)

        raw_data = {}
        for key, candidates in CHANNEL_MAP.items():
            for name in candidates:
                match = name if name in ch_labels else ch_lower.get(name.lower())
                if match:
                    idx = ch_labels.index(match)
                    raw_data[key] = (f.readSignal(idx),
                                     int(f.getSampleFrequency(idx)))
                    break
        f.close()
    except Exception as e:
        result["error"] = f"EDF read error: {e}"
        return result

    # resample
    max_time      = max(len(s)/fs for s,fs in raw_data.values())
    target_length = int(max_time * TARGET_FS)
    resampled     = {}

    for key, (sig, fs) in raw_data.items():
        if fs == TARGET_FS:
            res = sig.copy()
        else:
            res = resample(sig, int(len(sig) * TARGET_FS / fs))
        if len(res) < target_length:
            res = np.pad(res, (0, target_length - len(res)), mode="edge")
        else:
            res = res[:target_length]
        resampled[key] = res.astype(np.float32)

    del raw_data; gc.collect()

    fixed = {}
    for key in FIXED_CHANNEL_KEYS:
        fixed[key] = resampled.get(key, np.zeros(target_length, dtype=np.float32))

    del resampled; gc.collect()

    # reload events
    try:
        df = pd.read_excel(xlsx_path, sheet_name=SHEET_NAME, skiprows=SKIP_ROWS)
        df.columns = ["start","end","duration","event","stage"] + list(df.columns[5:])
        df = df[df["event"].notna() & (df["event"] != "Event name")].reset_index(drop=True)
        df["start"] = pd.to_datetime(df["start"], errors="coerce").dt.time
        df["end"]   = pd.to_datetime(df["end"],   errors="coerce").dt.time
        df = df.dropna(subset=["start","end"])
        df["start_sec"] = df["start"].apply(time_to_sec)
        df["end_sec"]   = df["end"].apply(time_to_sec)
        df.loc[df["start_sec"] < edf_start_sec, "start_sec"] += 86400
        df.loc[df["end_sec"]   < edf_start_sec, "end_sec"]   += 86400
        df["start_rel"] = df["start_sec"] - edf_start_sec
        df["end_rel"]   = df["end_sec"]   - edf_start_sec
        mask   = df["event"].str.contains("|".join(APNEA_KEYWORDS), case=False, na=False)
        events = df[mask].copy().reset_index(drop=True)
        del df
    except Exception as e:
        result["error"] = f"XLSX error: {e}"
        return result

    ev_start = events["start_rel"].to_numpy()
    ev_end   = events["end_rel"].to_numpy()

    def label_window(ws, we):
        if len(ev_start) == 0: return 0
        return int(np.any(~((ev_end <= ws) | (ev_start >= we))))

    # recompute windows
    total_dur = target_length / TARGET_FS
    offset    = next_aligned_second(edf_start_sec, WINDOW_SEC) - edf_start_sec

    X_recomp  = []
    y_recomp  = []
    times     = []
    current   = offset

    while current + WINDOW_SEC <= total_dur:
        s    = int(current * TARGET_FS)
        e    = s + SAMPLES_PER_WINDOW
        segs = [fixed[k][s:e] for k in FIXED_CHANNEL_KEYS]
        if all(len(sg) == SAMPLES_PER_WINDOW for sg in segs):
            X_recomp.append(np.stack(segs, axis=0))
            y_recomp.append(label_window(current, current + WINDOW_SEC))
            times.append(current)
        current += WINDOW_SEC

    del fixed, events; gc.collect()

    if not X_recomp:
        result["error"] = "No windows recomputed"
        return result

    X_rec = np.array(X_recomp, dtype=np.float32)
    y_rec = np.array(y_recomp, dtype=np.int8)
    result["n_recomputed"] = len(y_rec)

    del X_recomp, y_recomp; gc.collect()

    # ── CHECK 2: window count ─────────────────────────────────────
    result["n_windows_ok"] = (len(y_saved) == len(y_rec))

    n_compare = min(len(y_saved), len(y_rec))

    # ── CHECK 3: label match ──────────────────────────────────────
    label_matches          = int(np.sum(y_saved[:n_compare] == y_rec[:n_compare]))
    result["n_label_match"]  = label_matches
    result["label_match_pct"] = 100.0 * label_matches / n_compare if n_compare > 0 else 0
    result["labels_ok"]      = result["label_match_pct"] >= 99.0

    # ── CHECK 4: signal correlation (sample 10 random windows) ───
    n_sample = min(10, n_compare)
    sample_idx = np.random.default_rng(42).choice(n_compare, n_sample, replace=False)
    corrs = []
    for i in sample_idx:
        s_saved = X_saved[i].flatten()
        s_rec   = X_rec[i].flatten()
        if s_saved.std() > 1e-8 and s_rec.std() > 1e-8:
            corr = float(np.corrcoef(s_saved, s_rec)[0, 1])
            corrs.append(corr)

    result["signal_corr_mean"] = float(np.mean(corrs)) if corrs else 0.0
    result["signal_corr_ok"]   = result["signal_corr_mean"] >= 0.99

    # ── CHECK 5: timestamps in meta CSV ──────────────────────────
    if meta_path and os.path.exists(meta_path):
        meta = pd.read_csv(meta_path)
        if len(meta) == len(y_rec):
            expected_starts = [offset + i * WINDOW_SEC for i in range(len(y_rec))]
            saved_starts    = meta["start_rel_s"].tolist()
            ts_ok = all(abs(a - b) < 0.1 for a, b in
                        zip(expected_starts[:n_compare], saved_starts[:n_compare]))
            result["timestamps_ok"] = ts_ok
        else:
            result["timestamps_ok"] = False
    else:
        result["timestamps_ok"] = None   # no meta file to check

    del X_saved, X_rec, y_saved, y_rec; gc.collect()
    return result

# ──────────────────────────────────────────────────────────────────────────────
# 6.  RUN VERIFICATION ON ALL PATIENTS
# ──────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print(f"  VERIFYING {len(in_both)} patients")
print("=" * 65)

all_results = []
passed      = []
failed      = []

for pid in in_both:
    r = verify_patient(pid)
    all_results.append(r)

    checks = [r["shape_ok"], r["n_windows_ok"],
              r["labels_ok"], r["signal_corr_ok"]]

    all_pass = all(c for c in checks if c is not None) and r["error"] == ""

    status = "✅ PASS" if all_pass else "❌ FAIL"
    if all_pass:
        passed.append(pid)
    else:
        failed.append(pid)

    print(f"  {status} [{pid}]  "
          f"windows={r['n_saved']}/{r['n_recomputed']}  "
          f"label_match={r['label_match_pct']:.1f}%  "
          f"signal_corr={r['signal_corr_mean']:.4f}  "
          f"shape={'✓' if r['shape_ok'] else '✗'}  "
          f"ts={'✓' if r['timestamps_ok'] else '✗' if r['timestamps_ok'] is False else '-'}"
          + (f"  ⚠️  {r['error']}" if r["error"] else ""))

# ──────────────────────────────────────────────────────────────────────────────
# 7.  DETAILED REPORT FOR FAILED PATIENTS
# ──────────────────────────────────────────────────────────────────────────────
if failed:
    print("\n" + "=" * 65)
    print("  FAILURE DETAILS")
    print("=" * 65)
    for r in all_results:
        if r["pid"] not in failed:
            continue
        print(f"\n  [{r['pid']}]")
        print(f"    shape_ok        : {r['shape_ok']}")
        print(f"    n_windows_ok    : {r['n_windows_ok']}  "
              f"(saved={r['n_saved']}  recomputed={r['n_recomputed']})")
        print(f"    labels_ok       : {r['labels_ok']}  "
              f"({r['label_match_pct']:.1f}% match,  "
              f"{r['n_label_match']}/{min(r['n_saved'],r['n_recomputed'])} windows)")
        print(f"    signal_corr_ok  : {r['signal_corr_ok']}  "
              f"(mean corr = {r['signal_corr_mean']:.4f})")
        print(f"    timestamps_ok   : {r['timestamps_ok']}")
        if r["error"]:
            print(f"    error           : {r['error']}")

# ──────────────────────────────────────────────────────────────────────────────
# 8.  SAVE VERIFICATION REPORT
# ──────────────────────────────────────────────────────────────────────────────
report_df = pd.DataFrame(all_results)
report_df.to_csv("/kaggle/working/verification_report.csv", index=False)
print(f"\n  Full report saved → /kaggle/working/verification_report.csv")

# ──────────────────────────────────────────────────────────────────────────────
# 9.  OVERALL SUMMARY
# ──────────────────────────────────────────────────────────────────────────────
total_v   = len(in_both)
n_pass    = len(passed)
n_fail    = len(failed)

avg_label = report_df["label_match_pct"].mean()
avg_corr  = report_df["signal_corr_mean"].mean()

print(f"""
  ╔══════════════════════════════════════════════════════════╗
  ║              VERIFICATION SUMMARY                        ║
  ╠══════════════════════════════════════════════════════════╣
  ║  Patients verified  : {total_v:<35}║
  ║  PASSED             : {n_pass:<35}║
  ║  FAILED             : {n_fail:<35}║
  ║  Avg label match    : {avg_label:<34.2f}%║
  ║  Avg signal corr    : {avg_corr:<35.4f}║
  ╠══════════════════════════════════════════════════════════╣
  ║  Only in EDF (no npy) : {str(only_src):<32}║
  ║  Only in NPY (no EDF) : {str(only_npy):<32}║
  ╠══════════════════════════════════════════════════════════╣
  ║  Report : /kaggle/working/verification_report.csv       ║
  ╚══════════════════════════════════════════════════════════╝
""")

if n_fail == 0:
    print("  ✅ ALL PATIENTS PASSED — dataset matches original EDF/XLSX files")
else:
    print(f"  ❌ {n_fail} patients failed — see details above and verification_report.csv")
    print(f"     Failed: {failed}")

In [ ]:
# =============================================================================
#  VISUAL COMPARISON — X_SN32.npy  vs  SN32.edf
# =============================================================================

# ──────────────────────────────────────────────────────────────────────────────
# 0.  DOWNLOADS
# ──────────────────────────────────────────────────────────────────────────────
import kagglehub
src_path = kagglehub.dataset_download("nishantomhare/nimhanspatient-dataset")
npy_path = kagglehub.dataset_download("nishantomhare/detection-of-disordered-breathing-dataset2")
print("EDF/XLSX path :", src_path)
print("NPY path      :", npy_path)

# ──────────────────────────────────────────────────────────────────────────────
# 1.  IMPORTS
# ──────────────────────────────────────────────────────────────────────────────
import os, gc, math, warnings
import numpy as np
import pandas as pd
import pyedflib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.signal import resample
from collections import Counter
warnings.filterwarnings("ignore")

# ──────────────────────────────────────────────────────────────────────────────
# 2.  CONFIGURATION
# ──────────────────────────────────────────────────────────────────────────────
PID                = "SN32"
WINDOWS_TO_PLOT    = 3
TARGET_FS          = 200
WINDOW_SEC         = 30
SAMPLES_PER_WINDOW = WINDOW_SEC * TARGET_FS       # 6000
SHEET_NAME         = "Flow Events"
SKIP_ROWS          = 5
OUT_DIR            = "/kaggle/working"

FIXED_CHANNEL_KEYS = ["flow","chest","abd","spo2","pleth","ecg","pulse","flow2"]

CHANNEL_MAP = {
    "flow"  : ["Pressure Flow", "Airflow"],
    "chest" : ["Thorax", "Chest", "RIP Thorax"],
    "abd"   : ["RIP Abdomen", "Abdomen"],
    "spo2"  : ["SpO2", "SPO2", "SaO2"],
    "pleth" : ["Pleth", "Plethysmography"],
    "ecg"   : ["ECG 2", "ECG", "EKG"],
    "pulse" : ["Pulse", "Pulse Rate"],
    "flow2"   : ["Flow Th"],
}

APNEA_KEYWORDS = ["apnea", "hypopnea","RERA"]
NUM_CHANNELS   = len(FIXED_CHANNEL_KEYS)

# ──────────────────────────────────────────────────────────────────────────────
# 3.  HELPERS
# ──────────────────────────────────────────────────────────────────────────────
def time_to_sec(t):
    return t.hour * 3600 + t.minute * 60 + t.second

def next_aligned_second(start_sec, window):
    return math.ceil(start_sec / window) * window

def find_file(pid, prefix, ext, search_path):
    for root, _, files in os.walk(search_path):
        for f in files:
            if f == f"{prefix}_{pid}.{ext}":
                return os.path.join(root, f)
    return None

# ──────────────────────────────────────────────────────────────────────────────
# 4.  LOAD SAVED NPY
# ──────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"  Loading saved NPY for {PID}")
print(f"{'='*60}")

X_path    = find_file(PID, "X", "npy", npy_path)
y_path    = find_file(PID, "y", "npy", npy_path)
meta_path = find_file(PID, "meta", "csv", npy_path)

if not X_path:
    raise FileNotFoundError(f"X_{PID}.npy not found under {npy_path}")

X_npy = np.load(X_path).astype(np.float32)   # (N, 8, 6000)
y_npy = np.load(y_path)
meta  = pd.read_csv(meta_path) if meta_path else None

print(f"  X_npy shape : {X_npy.shape}")
print(f"  y_npy shape : {y_npy.shape}")
dist  = Counter(y_npy.tolist())
total = len(y_npy)
print(f"  Normal (0)  : {dist.get(0,0)} ({100*dist.get(0,0)/total:.1f}%)")
print(f"  Apnea  (1)  : {dist.get(1,0)} ({100*dist.get(1,0)/total:.1f}%)")

# ──────────────────────────────────────────────────────────────────────────────
# 5.  RECOMPUTE FROM EDF
# ──────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"  Recomputing from EDF for {PID}")
print(f"{'='*60}")

edf_path  = os.path.join(src_path, f"{PID}.edf")
xlsx_path = os.path.join(src_path, f"{PID}.xlsx")

f             = pyedflib.EdfReader(edf_path)
ch_labels     = f.getSignalLabels()
ch_lower      = {c.lower(): c for c in ch_labels}
edf_start     = pd.to_datetime(f.getStartdatetime()).time()
edf_start_sec = time_to_sec(edf_start)
print(f"  EDF start    : {edf_start}")
print(f"  EDF channels : {ch_labels}")

raw_data = {}
for key, candidates in CHANNEL_MAP.items():
    for name in candidates:
        match = name if name in ch_labels else ch_lower.get(name.lower())
        if match:
            idx = ch_labels.index(match)
            raw_data[key] = (f.readSignal(idx), int(f.getSampleFrequency(idx)))
            print(f"  ✅ {key:6s} → '{match}'")
            break
    else:
        print(f"  ⚠️  {key:6s} → NOT FOUND (zeros)")
f.close()

max_time      = max(len(s)/fs for s,fs in raw_data.values())
target_length = int(max_time * TARGET_FS)
resampled     = {}

for key, (sig, fs) in raw_data.items():
    res = sig.copy() if fs == TARGET_FS else resample(sig, int(len(sig)*TARGET_FS/fs))
    if len(res) < target_length:
        res = np.pad(res, (0, target_length - len(res)), mode="edge")
    else:
        res = res[:target_length]
    resampled[key] = res.astype(np.float32)

del raw_data; gc.collect()

fixed = {key: resampled.get(key, np.zeros(target_length, dtype=np.float32))
         for key in FIXED_CHANNEL_KEYS}
del resampled; gc.collect()

# load events
df = pd.read_excel(xlsx_path, sheet_name=SHEET_NAME, skiprows=SKIP_ROWS)
df.columns = ["start","end","duration","event","stage"] + list(df.columns[5:])
df = df[df["event"].notna() & (df["event"] != "Event name")].reset_index(drop=True)
df["start"] = pd.to_datetime(df["start"], errors="coerce").dt.time
df["end"]   = pd.to_datetime(df["end"],   errors="coerce").dt.time
df = df.dropna(subset=["start","end"])
df["start_sec"] = df["start"].apply(time_to_sec)
df["end_sec"]   = df["end"].apply(time_to_sec)
df.loc[df["start_sec"] < edf_start_sec, "start_sec"] += 86400
df.loc[df["end_sec"]   < edf_start_sec, "end_sec"]   += 86400
df["start_rel"] = df["start_sec"] - edf_start_sec
df["end_rel"]   = df["end_sec"]   - edf_start_sec
events = df[df["event"].str.contains("|".join(APNEA_KEYWORDS),
             case=False, na=False)].copy().reset_index(drop=True)
del df

ev_start = events["start_rel"].to_numpy()
ev_end   = events["end_rel"].to_numpy()

def label_window(ws, we):
    if len(ev_start) == 0: return 0
    return int(np.any(~((ev_end <= ws) | (ev_start >= we))))

# rebuild windows
total_dur  = target_length / TARGET_FS
offset     = next_aligned_second(edf_start_sec, WINDOW_SEC) - edf_start_sec
X_edf_list, y_edf_list, times = [], [], []
current = offset

while current + WINDOW_SEC <= total_dur:
    s    = int(current * TARGET_FS)
    e    = s + SAMPLES_PER_WINDOW
    segs = [fixed[k][s:e] for k in FIXED_CHANNEL_KEYS]
    if all(len(sg) == SAMPLES_PER_WINDOW for sg in segs):
        X_edf_list.append(np.stack(segs, axis=0))
        y_edf_list.append(label_window(current, current + WINDOW_SEC))
        times.append(current)
    current += WINDOW_SEC

X_edf = np.array(X_edf_list, dtype=np.float32)
y_edf = np.array(y_edf_list, dtype=np.int8)
del X_edf_list, fixed; gc.collect()

n_compare  = min(len(y_npy), len(y_edf))
label_acc  = 100 * np.mean(y_npy[:n_compare] == y_edf[:n_compare])
n_mismatch = int(np.sum(y_npy[:n_compare] != y_edf[:n_compare]))

print(f"\n  Recomputed X_edf : {X_edf.shape}")
print(f"  Saved      X_npy : {X_npy.shape}")
print(f"  Label match      : {label_acc:.2f}%")
print(f"  Mismatches       : {n_mismatch} / {n_compare}")

# ──────────────────────────────────────────────────────────────────────────────
# 6.  PLOT 1 — FULL RECORDING OVERVIEW
#     Use window-concatenated signals so lengths always match
# ──────────────────────────────────────────────────────────────────────────────
print("\n  Generating plots...")

# concatenate only the windows that exist in BOTH arrays
n_both    = min(len(X_edf), len(X_npy))
sig_edf   = np.concatenate([X_edf[w] for w in range(n_both)], axis=1)  # (8, N*6000)
sig_npy   = np.concatenate([X_npy[w] for w in range(n_both)], axis=1)  # (8, N*6000)
n_pts     = sig_edf.shape[1]
t_full    = np.arange(n_pts) / TARGET_FS / 60.0    # minutes — same length for both

# build apnea shading windows (in minutes)
apnea_spans = [(row["start_rel"]/60, row["end_rel"]/60)
               for _, row in events.iterrows()]

fig, axes = plt.subplots(NUM_CHANNELS, 1,
                          figsize=(24, 2.8 * NUM_CHANNELS), sharex=True)
fig.suptitle(
    f"{PID} — Full Recording  |  EDF resampled (blue) vs NPY saved (red dashed)\n"
    f"Windows: EDF={len(X_edf)}  NPY={len(X_npy)}  Compared={n_both}  "
    f"Label match={label_acc:.2f}%  Mismatches={n_mismatch}",
    fontsize=12, fontweight="bold", y=1.005)

for i, key in enumerate(FIXED_CHANNEL_KEYS):
    ax = axes[i]

    ax.plot(t_full, sig_edf[i], color="steelblue",
            lw=0.4, alpha=0.85, label="EDF resampled")
    ax.plot(t_full, sig_npy[i], color="crimson",
            lw=0.4, alpha=0.7,  linestyle="--", label="NPY saved")

    for (ts, te) in apnea_spans:
        ax.axvspan(ts, te, alpha=0.10, color="orange", zorder=0)

    ax.set_ylabel(key, fontsize=9, fontweight="bold")
    ax.tick_params(axis="y", labelsize=7)
    ax.grid(True, alpha=0.25)
    if i == 0:
        ax.legend(loc="upper right", fontsize=8)

axes[-1].set_xlabel("Time (minutes)", fontsize=10)
plt.tight_layout()
out1 = os.path.join(OUT_DIR, f"{PID}_full_recording_comparison.png")
plt.savefig(out1, dpi=120, bbox_inches="tight")
plt.close()
print(f"  ✅ Plot 1 saved : {out1}")

del sig_edf, sig_npy; gc.collect()

# ──────────────────────────────────────────────────────────────────────────────
# 7.  PLOT 2 — WINDOW DETAIL (3 normal + 3 apnea)
# ──────────────────────────────────────────────────────────────────────────────
normal_wins = np.where(y_npy[:n_both] == 0)[0][:WINDOWS_TO_PLOT].tolist()
apnea_wins  = np.where(y_npy[:n_both] == 1)[0][:WINDOWS_TO_PLOT].tolist()
sel_windows = normal_wins + apnea_wins

t_win = np.arange(SAMPLES_PER_WINDOW) / TARGET_FS   # 0..30 seconds

for win_idx in sel_windows:
    lbl_npy  = int(y_npy[win_idx])
    lbl_edf  = int(y_edf[win_idx]) if win_idx < len(y_edf) else -1
    match    = "✓ MATCH" if lbl_npy == lbl_edf else "✗ MISMATCH"
    win_type = "APNEA" if lbl_npy == 1 else "NORMAL"
    t_start  = times[win_idx] if win_idx < len(times) else win_idx * WINDOW_SEC

    fig = plt.figure(figsize=(20, 2.6 * NUM_CHANNELS))
    fig.suptitle(
        f"{PID}  Window #{win_idx}  |  {win_type}  |  "
        f"NPY label={lbl_npy}  EDF label={lbl_edf}  {match}\n"
        f"Window start: {t_start:.1f}s  ({t_start/60:.2f} min)",
        fontsize=11, fontweight="bold")

    gs = gridspec.GridSpec(NUM_CHANNELS, 1, hspace=0.06)

    for i, key in enumerate(FIXED_CHANNEL_KEYS):
        ax = fig.add_subplot(gs[i])

        s_edf = X_edf[win_idx, i, :]
        s_npy = X_npy[win_idx, i, :]

        corr = (float(np.corrcoef(s_edf, s_npy)[0, 1])
                if s_edf.std() > 1e-8 and s_npy.std() > 1e-8 else float("nan"))

        ax.plot(t_win, s_edf, color="steelblue", lw=1.0,
                alpha=0.9, label="EDF")
        ax.plot(t_win, s_npy, color="crimson",   lw=1.0,
                alpha=0.85, linestyle="--", label="NPY")
        ax.fill_between(t_win, s_edf, s_npy,
                        alpha=0.15, color="purple", label="diff")

        ax.set_ylabel(f"{key}\ncorr={corr:.4f}", fontsize=8, fontweight="bold")
        ax.tick_params(axis="y", labelsize=7)
        ax.grid(True, alpha=0.25)

        if i == 0:
            ax.legend(loc="upper right", fontsize=8, ncol=3)
        if i < NUM_CHANNELS - 1:
            ax.set_xticklabels([])

    fig.axes[-1].set_xlabel("Time within window (seconds)", fontsize=10)
    fname    = f"{PID}_win{win_idx:04d}_{win_type.lower()}.png"
    out_path = os.path.join(OUT_DIR, fname)
    plt.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.close()
    print(f"  ✅ Plot 2 — Window {win_idx:4d} ({win_type:6s}) saved : {fname}")

# ──────────────────────────────────────────────────────────────────────────────
# 8.  PLOT 3 — LABEL TIMELINE
# ──────────────────────────────────────────────────────────────────────────────
win_nums  = np.arange(n_compare)
mismatch  = (y_npy[:n_compare] != y_edf[:n_compare]).astype(int)

fig, axes = plt.subplots(3, 1, figsize=(22, 7), sharex=True)
fig.suptitle(f"{PID} — Label Timeline Comparison", fontsize=12, fontweight="bold")

axes[0].step(win_nums, y_npy[:n_compare], where="mid",
             color="crimson", lw=1.2, label="NPY labels")
axes[0].set_yticks([0,1]); axes[0].set_yticklabels(["Normal","Apnea"])
axes[0].set_ylabel("NPY", fontsize=9, fontweight="bold")
axes[0].legend(loc="upper right", fontsize=8); axes[0].grid(True, alpha=0.25)

axes[1].step(win_nums, y_edf[:n_compare], where="mid",
             color="steelblue", lw=1.2, label="EDF recomputed")
axes[1].set_yticks([0,1]); axes[1].set_yticklabels(["Normal","Apnea"])
axes[1].set_ylabel("EDF", fontsize=9, fontweight="bold")
axes[1].legend(loc="upper right", fontsize=8); axes[1].grid(True, alpha=0.25)

axes[2].bar(win_nums, mismatch, color="red", width=1.0)
axes[2].set_ylabel("Mismatch", fontsize=9, fontweight="bold")
axes[2].set_xlabel("Window index", fontsize=10)
axes[2].set_title(f"Total mismatches: {n_mismatch}/{n_compare} "
                  f"({100*n_mismatch/n_compare:.2f}%)", fontsize=9)
axes[2].grid(True, alpha=0.25)

plt.tight_layout()
out3 = os.path.join(OUT_DIR, f"{PID}_label_timeline.png")
plt.savefig(out3, dpi=120, bbox_inches="tight")
plt.close()
print(f"  ✅ Plot 3 saved : {out3}")

# ──────────────────────────────────────────────────────────────────────────────
# 9.  PLOT 4 — PER-CHANNEL CORRELATION HEATMAP
# ──────────────────────────────────────────────────────────────────────────────
N_SAMPLE    = min(100, n_both)
sample_idx  = np.linspace(0, n_both-1, N_SAMPLE, dtype=int)
corr_matrix = np.zeros((NUM_CHANNELS, N_SAMPLE))

for j, wi in enumerate(sample_idx):
    for i in range(NUM_CHANNELS):
        s1, s2 = X_edf[wi, i, :], X_npy[wi, i, :]
        if s1.std() > 1e-8 and s2.std() > 1e-8:
            corr_matrix[i, j] = np.corrcoef(s1, s2)[0, 1]
        else:
            corr_matrix[i, j] = 1.0 if np.allclose(s1, s2) else 0.0

fig, ax = plt.subplots(figsize=(20, 5))
im = ax.imshow(corr_matrix, aspect="auto", vmin=-1, vmax=1,
               cmap="RdYlGn", interpolation="nearest")
plt.colorbar(im, ax=ax, label="Pearson Correlation")
ax.set_yticks(range(NUM_CHANNELS))
ax.set_yticklabels(FIXED_CHANNEL_KEYS, fontsize=10, fontweight="bold")
ax.set_xlabel("Sampled window index", fontsize=10)
ax.set_title(f"{PID} — Per-Channel Signal Correlation Heatmap  "
             f"({N_SAMPLE} sampled windows)\nGreen=1.0 perfect, Red=mismatch",
             fontsize=11)

for i in range(NUM_CHANNELS):
    mean_c = corr_matrix[i].mean()
    color  = "darkgreen" if mean_c > 0.99 else "darkred"
    ax.text(N_SAMPLE + 0.5, i, f"μ={mean_c:.4f}",
            va="center", ha="left", fontsize=8, color=color)

plt.tight_layout()
out4 = os.path.join(OUT_DIR, f"{PID}_correlation_heatmap.png")
plt.savefig(out4, dpi=120, bbox_inches="tight")
plt.close()
print(f"  ✅ Plot 4 saved : {out4}")

# ──────────────────────────────────────────────────────────────────────────────
# 10.  SUMMARY
# ──────────────────────────────────────────────────────────────────────────────
print(f"""
  ╔══════════════════════════════════════════════════════════╗
  ║     {PID}  NPY vs EDF COMPARISON COMPLETE              ║
  ╠══════════════════════════════════════════════════════════╣
  ║  X_npy shape    : {str(X_npy.shape):<38}║
  ║  X_edf shape    : {str(X_edf.shape):<38}║
  ║  Label match    : {f"{label_acc:.2f}%":<38}║
  ║  Mismatches     : {f"{n_mismatch} / {n_compare}":<38}║
  ╠══════════════════════════════════════════════════════════╣
  ║  Saved to /kaggle/working/ :                            ║
  ║  1. {PID}_full_recording_comparison.png                 ║
  ║  2. {PID}_win####_normal/apnea.png  (6 windows)        ║
  ║  3. {PID}_label_timeline.png                            ║
  ║  4. {PID}_correlation_heatmap.png                       ║
  ╚══════════════════════════════════════════════════════════╝
""")

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

OUT_DIR = "/kaggle/working"
PID     = "SN32"

# collect all comparison images for this patient
img_files = sorted([
    f for f in os.listdir(OUT_DIR)
    if f.startswith(PID) and f.endswith(".png")
])

print(f"Found {len(img_files)} images for {PID}:")
for f in img_files:
    size_mb = os.path.getsize(os.path.join(OUT_DIR, f)) / 1e6
    print(f"  {f}  ({size_mb:.1f} MB)")

# display each image full size
for fname in img_files:
    img_path = os.path.join(OUT_DIR, fname)
    img      = mpimg.imread(img_path)

    h, w  = img.shape[:2]
    fig_w = min(24, w / 80)
    fig_h = fig_w * h / w

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(fname, fontsize=10, pad=6)
    plt.tight_layout()
    plt.show()
    print(f"  ✅ Displayed: {fname}  ({w}×{h} px)")

In [ ]:
from IPython.display import display, Image as IPImage
import os

OUT_DIR = "/kaggle/working"
PID     = "SN32"

# collect all images for this patient
img_files = sorted([
    f for f in os.listdir(OUT_DIR)
    if f.startswith(PID) and f.endswith(".png")
])

print(f"Found {len(img_files)} images:\n")

for fname in img_files:
    path = os.path.join(OUT_DIR, fname)
    print(f"─── {fname} ───")
    display(IPImage(filename=path))
    print()

In [ ]:
import kagglehub
import numpy as np
import pandas as pd

# download dataset
npy_path = kagglehub.dataset_download("nishantomhare/detection-of-disordered-breathing-dataset")

import os
def find_file(pid, prefix, ext, search_path):
    for root, _, files in os.walk(search_path):
        for f in files:
            if f == f"{prefix}_{pid}.{ext}":
                return os.path.join(root, f)
    return None

PID = "SN8"

y_path    = find_file(PID, "y",    "npy", npy_path)
meta_path = find_file(PID, "meta", "csv", npy_path)

y    = np.load(y_path)
meta = pd.read_csv(meta_path)

# add human-readable status column
meta["status"] = meta["label"].map({0: "🟢 NORMAL", 1: "🔴 APNEA"})

print(f"Patient      : {PID}")
print(f"Total windows: {len(meta)}")
print(f"Normal  (0)  : {(y==0).sum()} ({100*(y==0).mean():.1f}%)")
print(f"Apnea   (1)  : {(y==1).sum()} ({100*(y==1).mean():.1f}%)")
print()

# print every window
print(f"{'WIN#':>5}  {'START':>10}  {'END':>10}  {'START_REL_S':>12}  {'END_REL_S':>10}  {'STATUS'}")
print("─" * 75)
for _, row in meta.iterrows():
    print(f"{int(row.window_idx):>5}  "
          f"{row.start_clock:>10}  "
          f"{row.end_clock:>10}  "
          f"{row.start_rel_s:>12.1f}  "
          f"{row.end_rel_s:>10.1f}  "
          f"{row.status}")

In [ ]:
import kagglehub
import numpy as np
import pandas as pd

# download dataset
npy_path = kagglehub.dataset_download("nishantomhare/detection-of-disordered-breathing-dataset2")

import os
def find_file(pid, prefix, ext, search_path):
    for root, _, files in os.walk(search_path):
        for f in files:
            if f == f"{prefix}_{pid}.{ext}":
                return os.path.join(root, f)
    return None

PID = "SN24"

y_path    = find_file(PID, "y",    "npy", npy_path)
meta_path = find_file(PID, "meta", "csv", npy_path)

y    = np.load(y_path)
meta = pd.read_csv(meta_path)

# add human-readable status column
meta["status"] = meta["label"].map({0: "🟢 NORMAL", 1: "🔴 APNEA"})

print(f"Patient      : {PID}")
print(f"Total windows: {len(meta)}")
print(f"Normal  (0)  : {(y==0).sum()} ({100*(y==0).mean():.1f}%)")
print(f"Apnea   (1)  : {(y==1).sum()} ({100*(y==1).mean():.1f}%)")
print()

# print every window
print(f"{'WIN#':>5}  {'START':>10}  {'END':>10}  {'START_REL_S':>12}  {'END_REL_S':>10}  {'STATUS'}")
print("─" * 75)
for _, row in meta.iterrows():
    print(f"{int(row.window_idx):>5}  "
          f"{row.start_clock:>10}  "
          f"{row.end_clock:>10}  "
          f"{row.start_rel_s:>12.1f}  "
          f"{row.end_rel_s:>10.1f}  "
          f"{row.status}")

In [ ]:
import kagglehub
import numpy as np
import pandas as pd

# download dataset
npy_path = kagglehub.dataset_download("nishantomhare/detection-of-disordered-breathing-dataset2")

import os
def find_file(pid, prefix, ext, search_path):
    for root, _, files in os.walk(search_path):
        for f in files:
            if f == f"{prefix}_{pid}.{ext}":
                return os.path.join(root, f)
    return None

PID = "SN23"

y_path    = find_file(PID, "y",    "npy", npy_path)
meta_path = find_file(PID, "meta", "csv", npy_path)

y    = np.load(y_path)
meta = pd.read_csv(meta_path)

# add human-readable status column
meta["status"] = meta["label"].map({0: "🟢 NORMAL", 1: "🔴 APNEA"})

print(f"Patient      : {PID}")
print(f"Total windows: {len(meta)}")
print(f"Normal  (0)  : {(y==0).sum()} ({100*(y==0).mean():.1f}%)")
print(f"Apnea   (1)  : {(y==1).sum()} ({100*(y==1).mean():.1f}%)")
print()

# print every window
print(f"{'WIN#':>5}  {'START':>10}  {'END':>10}  {'START_REL_S':>12}  {'END_REL_S':>10}  {'STATUS'}")
print("─" * 75)
for _, row in meta.iterrows():
    print(f"{int(row.window_idx):>5}  "
          f"{row.start_clock:>10}  "
          f"{row.end_clock:>10}  "
          f"{row.start_rel_s:>12.1f}  "
          f"{row.end_rel_s:>10.1f}  "
          f"{row.status}")

In [ ]:
import kagglehub
import numpy as np
import pandas as pd

# download dataset
npy_path = kagglehub.dataset_download("nishantomhare/detection-of-disordered-breathing-dataset2")

import os
def find_file(pid, prefix, ext, search_path):
    for root, _, files in os.walk(search_path):
        for f in files:
            if f == f"{prefix}_{pid}.{ext}":
                return os.path.join(root, f)
    return None

PID = "SN22"

y_path    = find_file(PID, "y",    "npy", npy_path)
meta_path = find_file(PID, "meta", "csv", npy_path)

y    = np.load(y_path)
meta = pd.read_csv(meta_path)

# add human-readable status column
meta["status"] = meta["label"].map({0: "🟢 NORMAL", 1: "🔴 APNEA"})

print(f"Patient      : {PID}")
print(f"Total windows: {len(meta)}")
print(f"Normal  (0)  : {(y==0).sum()} ({100*(y==0).mean():.1f}%)")
print(f"Apnea   (1)  : {(y==1).sum()} ({100*(y==1).mean():.1f}%)")
print()

# print every window
print(f"{'WIN#':>5}  {'START':>10}  {'END':>10}  {'START_REL_S':>12}  {'END_REL_S':>10}  {'STATUS'}")
print("─" * 75)
for _, row in meta.iterrows():
    print(f"{int(row.window_idx):>5}  "
          f"{row.start_clock:>10}  "
          f"{row.end_clock:>10}  "
          f"{row.start_rel_s:>12.1f}  "
          f"{row.end_rel_s:>10.1f}  "
          f"{row.status}")

In [ ]:
import kagglehub
import numpy as np
import pandas as pd

# download dataset
npy_path = kagglehub.dataset_download("nishantomhare/detection-of-disordered-breathing-dataset2")

import os
def find_file(pid, prefix, ext, search_path):
    for root, _, files in os.walk(search_path):
        for f in files:
            if f == f"{prefix}_{pid}.{ext}":
                return os.path.join(root, f)
    return None

PID = "SN1"

y_path    = find_file(PID, "y",    "npy", npy_path)
meta_path = find_file(PID, "meta", "csv", npy_path)

y    = np.load(y_path)
meta = pd.read_csv(meta_path)

# add human-readable status column
meta["status"] = meta["label"].map({0: "🟢 NORMAL", 1: "🔴 APNEA"})

print(f"Patient      : {PID}")
print(f"Total windows: {len(meta)}")
print(f"Normal  (0)  : {(y==0).sum()} ({100*(y==0).mean():.1f}%)")
print(f"Apnea   (1)  : {(y==1).sum()} ({100*(y==1).mean():.1f}%)")
print()

# print every window
print(f"{'WIN#':>5}  {'START':>10}  {'END':>10}  {'START_REL_S':>12}  {'END_REL_S':>10}  {'STATUS'}")
print("─" * 75)
for _, row in meta.iterrows():
    print(f"{int(row.window_idx):>5}  "
          f"{row.start_clock:>10}  "
          f"{row.end_clock:>10}  "
          f"{row.start_rel_s:>12.1f}  "
          f"{row.end_rel_s:>10.1f}  "
          f"{row.status}")

In [ ]:
import kagglehub
import os
import numpy as np
import pandas as pd
from collections import Counter

# download dataset
npy_path = kagglehub.dataset_download("nishantomhare/detection-of-disordered-breathing-dataset2")
print("NPY path:", npy_path)

# ──────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ──────────────────────────────────────────────────────────────────────────────
FIXED_CHANNEL_KEYS = ["flow","chest","abd","spo2","pleth","ecg","pulse","flow2"]
ALL_PIDS           = [f"SN{i}" for i in range(1, 101)]
SAMPLES_PER_WINDOW = 6000

def find_file(pid, prefix, ext, search_path):
    for root, _, files in os.walk(search_path):
        for f in files:
            if f == f"{prefix}_{pid}.{ext}":
                return os.path.join(root, f)
    return None

# ──────────────────────────────────────────────────────────────────────────────
# BUILD STATS FOR EACH PATIENT
# ──────────────────────────────────────────────────────────────────────────────
rows = []

print(f"\n{'='*75}")
print(f"  SCANNING ALL PATIENTS SN1 → SN100")
print(f"{'='*75}")

for pid in ALL_PIDS:
    X_path    = find_file(pid, "X",    "npy", npy_path)
    y_path    = find_file(pid, "y",    "npy", npy_path)
    meta_path = find_file(pid, "meta", "csv", npy_path)

    row = {
        "patient"           : pid,
        "file_present"      : False,
        "total_windows"     : 0,
        "normal_windows"    : 0,
        "apnea_windows"     : 0,
        "apnea_pct"         : 0.0,
        "total_samples"     : 0,
        "n_channels"        : 0,
        "zero_padded_ch"    : "",
        "present_channels"  : "",
        "missing_channels"  : "",
        "dtype"             : "",
        "X_size_mb"         : 0.0,
        "duration_min"      : 0.0,
        "status"            : "",
    }

    # ── missing entirely ─────────────────────────────────────────
    if not X_path or not y_path:
        row["status"] = "❌ FILE MISSING"
        rows.append(row)
        print(f"  ❌ [{pid}] FILES NOT FOUND")
        continue

    # ── load arrays ──────────────────────────────────────────────
    try:
        X = np.load(X_path, mmap_mode="r")
        y = np.load(y_path, mmap_mode="r")
    except Exception as e:
        row["status"] = f"❌ LOAD ERROR: {e}"
        rows.append(row)
        print(f"  ❌ [{pid}] Load error: {e}")
        continue

    dist = Counter(y.tolist())
    n_win   = len(y)
    n_apnea = dist.get(1, 0)
    n_norm  = dist.get(0, 0)

    row["file_present"]   = True
    row["total_windows"]  = n_win
    row["normal_windows"] = n_norm
    row["apnea_windows"]  = n_apnea
    row["apnea_pct"]      = round(100 * n_apnea / n_win, 1) if n_win > 0 else 0.0
    row["total_samples"]  = n_win * SAMPLES_PER_WINDOW
    row["n_channels"]     = X.shape[1] if X.ndim == 3 else 0
    row["dtype"]          = str(X.dtype)
    row["X_size_mb"]      = round(os.path.getsize(X_path) / 1e6, 1)
    row["duration_min"]   = round(n_win * 30 / 60, 1)

    # ── detect zero-padded (missing) channels ────────────────────
    # a channel is zero-padded if ALL values across ALL windows are 0
    zero_padded = []
    present_ch  = []
    missing_ch  = []

    if X.ndim == 3 and X.shape[1] == len(FIXED_CHANNEL_KEYS):
        for ci, ch_name in enumerate(FIXED_CHANNEL_KEYS):
            ch_data = X[:, ci, :]          # (N, 6000)
            if np.all(ch_data == 0):
                zero_padded.append(ch_name)
                missing_ch.append(ch_name)
            else:
                present_ch.append(ch_name)
    else:
        missing_ch = ["shape_mismatch"]

    row["zero_padded_ch"]   = ", ".join(zero_padded)  if zero_padded  else "none"
    row["present_channels"] = ", ".join(present_ch)   if present_ch   else "none"
    row["missing_channels"] = ", ".join(missing_ch)   if missing_ch   else "none"
    row["status"]           = "⚠️ HAS ZERO PADDING" if zero_padded else "✅ COMPLETE"

    del X, y

    print(f"  {row['status']} [{pid}]  "
          f"windows={n_win}  "
          f"apnea={n_apnea}({row['apnea_pct']}%)  "
          f"samples={row['total_samples']:,}  "
          f"dur={row['duration_min']}min  "
          f"size={row['X_size_mb']}MB  "
          + (f"zeros={zero_padded}" if zero_padded else ""))

    rows.append(row)

# ──────────────────────────────────────────────────────────────────────────────
# SAVE CSV
# ──────────────────────────────────────────────────────────────────────────────
df = pd.DataFrame(rows)
csv_path = "/kaggle/working/dataset_stats_SN1_SN100.csv"
df.to_csv(csv_path, index=False)
print(f"\n✅ Saved: {csv_path}")

# ──────────────────────────────────────────────────────────────────────────────
# PRINT SUMMARY TABLE
# ──────────────────────────────────────────────────────────────────────────────
present = df[df["file_present"] == True]
missing = df[df["file_present"] == False]
padded  = present[present["zero_padded_ch"] != "none"]

print(f"""
{'='*75}
  DATASET SUMMARY — SN1 to SN100
{'='*75}
  Total patients expected : 100
  Files present           : {len(present)}
  Files missing           : {len(missing)}
  Patients with zero pads : {len(padded)}

  Total windows           : {present['total_windows'].sum():,}
  Total apnea windows     : {present['apnea_windows'].sum():,}
  Total normal windows    : {present['normal_windows'].sum():,}
  Total samples           : {present['total_samples'].sum():,}
  Total duration          : {present['duration_min'].sum()/60:.1f} hours

  Avg apnea %             : {present['apnea_pct'].mean():.1f}%
  Avg windows/patient     : {present['total_windows'].mean():.0f}
  Total disk size         : {present['X_size_mb'].sum()/1024:.2f} GB
{'='*75}
""")

# missing patients
if len(missing) > 0:
    print(f"  ❌ MISSING PATIENTS ({len(missing)}):")
    for _, r in missing.iterrows():
        print(f"     {r['patient']}")

# patients with zero-padded channels
if len(padded) > 0:
    print(f"\n  ⚠️  PATIENTS WITH ZERO-PADDED CHANNELS ({len(padded)}):")
    print(f"  {'PATIENT':>8}  {'ZERO-PADDED CHANNELS'}")
    print("  " + "-"*40)
    for _, r in padded.iterrows():
        print(f"  {r['patient']:>8}  {r['zero_padded_ch']}")

# full table
print(f"\n  FULL TABLE:")
print(df[["patient","file_present","total_windows","apnea_windows",
          "normal_windows","apnea_pct","total_samples","duration_min",
          "zero_padded_ch","status"]].to_string(index=False))